<a href="https://colab.research.google.com/github/furalmiqbadi/PembelajaranMesin_01_AbdulGhofurAlmiqbadi/blob/main/JS04/JS04_Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Persiapan

In [30]:
import pandas as pd
import numpy as np

# load dataset
df = pd.read_csv('/content/insurance.csv')

## Identifikasi variabel-variabel yang akan digunakan sebagai variabel bebas (fitur) dan variabel target (biaya medis personal).

In [31]:
# ubah data teks (kategori) menjadi angka
df = pd.get_dummies(df, columns=['sex', 'smoker', 'region'], drop_first=True)

# pisahkan variabel bebas (X) dan variabel target (y)
X = df.drop('charges', axis=1)
y = df['charges']

# output
print("Variabel Bebas (Fitur):", list(X.columns))
print("Variabel Target:", y.name)

Variabel Bebas (Fitur): ['age', 'bmi', 'children', 'sex_male', 'smoker_yes', 'region_northwest', 'region_southeast', 'region_southwest']
Variabel Target: charges


## Bagi dataset menjadi data latih (train) dan data uji (test) dengan proporsi yang sesuai.

In [34]:
from sklearn.model_selection import train_test_split

# split 80:20 (Data Latih : Data Uji)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# output
print(f"Jumlah Data Latih: {X_train.shape[0]} baris")
print(f"Jumlah Data Uji: {X_test.shape[0]} baris")

Jumlah Data Latih: 1070 baris
Jumlah Data Uji: 268 baris


## Lakukan feature scaling jika diperlukan.

In [40]:
from sklearn.preprocessing import StandardScaler

# scaling fitur
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ouput
print("train", X_train_scaled)
print("test", X_test_scaled)

train [[ 0.47222651 -1.75652513  0.73433626 ...  1.78316783 -0.59966106
  -0.5723141 ]
 [ 0.54331294 -1.03308239 -0.91119211 ... -0.56079971 -0.59966106
  -0.5723141 ]
 [ 0.8987451  -0.94368672 -0.91119211 ... -0.56079971  1.66760869
  -0.5723141 ]
 ...
 [ 1.3252637  -0.89153925 -0.91119211 ... -0.56079971 -0.59966106
  -0.5723141 ]
 [-0.16755139  2.82086429  0.73433626 ... -0.56079971 -0.59966106
   1.74729228]
 [ 1.1120044  -0.10932713 -0.91119211 ... -0.56079971 -0.59966106
   1.74729228]]
test [[ 0.40114007 -0.89153925  0.73433626 ... -0.56079971 -0.59966106
  -0.5723141 ]
 [-0.23863782 -0.08946143 -0.91119211 ...  1.78316783 -0.59966106
  -0.5723141 ]
 [ 1.75178229 -0.60845296 -0.91119211 ...  1.78316783 -0.59966106
  -0.5723141 ]
 ...
 [-0.09646495 -0.41972876 -0.08842793 ... -0.56079971 -0.59966106
  -0.5723141 ]
 [ 1.04091797  2.78941026 -0.91119211 ... -0.56079971  1.66760869
  -0.5723141 ]
 [ 0.82765867  0.60252728 -0.08842793 ... -0.56079971 -0.59966106
   1.74729228]]


## Buat model multiple linear regression menggunakan Scikit-Learn.

In [42]:
from sklearn.linear_model import LinearRegression

# buat model multiple linear regression
mlr = LinearRegression()

# output
print("Model:", mlr)

Model: LinearRegression()


## Latih model pada data latih dan lakukan prediksi pada data uji.

In [45]:
# latih model & prediksi
mlr.fit(X_train_scaled, y_train)
y_pred_mlr = mlr.predict(X_test_scaled)

# output
print("Data Prediksi:", y_pred_mlr[:5])
print("Data Asli", y_test.values[:5])

Data Prediksi: [ 8969.55027444  7068.74744287 36858.41091155  9454.67850053
 26973.17345656]
Data Asli [ 9095.06825  5272.1758  29330.98315  9301.89355 33750.2918 ]


## Evaluasi model dengan menghitung metrik seperti R-squared, MSE, dan MAE. Tampilkan hasil evaluasi.

In [46]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# cek R2, MSE, MAE
r2_mlr = r2_score(y_test, y_pred_mlr)
mse_mlr = mean_squared_error(y_test, y_pred_mlr)
mae_mlr = mean_absolute_error(y_test, y_pred_mlr)

# output
print(f"R-squared: {r2_mlr:.4f}")
print(f"MSE: {mse_mlr:.2f}")
print(f"MAE: {mae_mlr:.2f}")

R-squared: 0.7836
MSE: 33596915.85
MAE: 4181.19


## Ulagi langkah 4 dengan menggunakan model SVR. Anda dapat bereksperimen dengan dengan melakukan hyperparameter tunning.

In [53]:
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV

# epsilon diperkecil (karena y sudah di-scaling), C diperbesar, dan gamma ikut dituning
param_grid = {
    'C': [10, 100, 1000, 10000],
    'epsilon': [0.001, 0.01, 0.05, 0.1],
    'gamma': ['scale', 0.01, 0.1]
}

# cv=5 agar evaluasi lebih stabil, n_jobs=-1 agar semua core CPU dipakai agar lebih cepat
grid = GridSearchCV(SVR(kernel='rbf'), param_grid, cv=5, scoring='r2', n_jobs=-1)
grid.fit(X_train_scaled, y_train)

print("Parameter Terbaik:", grid.best_params_)
print("Skor CV Terbaik:", round(grid.best_score_, 4))

# pakai estimator terbaik langsung untuk prediksi
svr = grid.best_estimator_
y_pred_svr = svr.predict(X_test_scaled)

r2_svr = r2_score(y_test, y_pred_svr)
mse_svr = mean_squared_error(y_test, y_pred_svr)
mae_svr = mean_absolute_error(y_test, y_pred_svr)

print(f"R-squared: {r2_svr:.4f}")
print(f"MSE: {mse_svr:.2f}")
print(f"MAE: {mae_svr:.2f}")

Parameter Terbaik: {'C': 10000, 'epsilon': 0.1, 'gamma': 0.1}
Skor CV Terbaik: 0.8199
R-squared: 0.8550
MSE: 22509459.25
MAE: 1808.80
